In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, Optional, Dict

TICKERS = ["AAPL", "TSLA", "MSFT"]
MIN_SAMPLES_WARN = 30


class RegimeParamsEstimator:
    def __init__(
        self,
        data_dir: Optional[Path] = None,
        state_path: Optional[Path] = None,
        stock_paths: Optional[Dict[str, Path]] = None,
        rf_gross: float = 1.0,
    ):
        if data_dir is None:
            data_dir = Path.cwd() / "data"
            if not data_dir.exists():
                data_dir = Path("/mnt/data")
        self.data_dir = Path(data_dir)
        self.rf_gross = float(rf_gross)
        self.state_path = Path(state_path) if state_path else self.data_dir / "sp500_market_state_2014_2023.csv"
        if stock_paths is None:
            stock_paths = {
                "AAPL": self.data_dir / "apple_stock.csv",
                "TSLA": self.data_dir / "tesla_stock.csv",
                "MSFT": self.data_dir / "microsoft_stock.csv",
            }
        self.stock_paths = {k: Path(v) for k, v in stock_paths.items()}
        self._merged = self._returns_df = self._z_aligned = self._counts = None

    def _load_and_merge(self) -> pd.DataFrame:
        state = pd.read_csv(self.state_path)
        date_col = "datetime" if "datetime" in state.columns else state.columns[0]
        bool_col = "bool" if "bool" in state.columns else state.columns[1]
        state[date_col] = pd.to_datetime(state[date_col])
        state = state.rename(columns={date_col: "date", bool_col: "z"})
        state = state[["date", "z"]].dropna().sort_values("date").reset_index(drop=True)
        dfs = [state]
        for ticker in TICKERS:
            path = self.stock_paths[ticker]
            df = pd.read_csv(path)
            dc = "Date" if "Date" in df.columns else df.columns[0]
            ac = "Adj Close" if "Adj Close" in df.columns else [c for c in df.columns if "adj" in c.lower() or "close" in c.lower()][-1]
            df[dc] = pd.to_datetime(df[dc])
            df = df[[dc, ac]].rename(columns={dc: "date", ac: f"adj_{ticker}"})
            df = df.dropna().sort_values("date").reset_index(drop=True)
            dfs.append(df)
        merged = dfs[0]
        for d in dfs[1:]:
            merged = merged.merge(d, on="date", how="inner")
        return merged.sort_values("date").reset_index(drop=True)

    def _compute_gross_returns_and_align_regime(self, merged: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        out = merged[["date", "z"]].copy()
        for ticker in TICKERS:
            out[f"gross_{ticker}"] = merged[f"adj_{ticker}"].shift(-1) / merged[f"adj_{ticker}"]
        out = out.iloc[:-1].copy()
        return out, out["z"]

    def _estimate_P(self, z: pd.Series) -> np.ndarray:
        z = z.astype(int)
        N = np.zeros((2, 2))
        for t in range(len(z) - 1):
            i, j = z.iloc[t], z.iloc[t + 1]
            if 0 <= i <= 1 and 0 <= j <= 1:
                N[i, j] += 1
        P = np.zeros((2, 2))
        for i in range(2):
            row_sum = N[i, :].sum()
            P[i, :] = (N[i, :] / row_sum) if row_sum > 0 else (np.eye(2)[i])
        return P

    def _estimate_R_low_R_high(self, df: pd.DataFrame, z: pd.Series) -> Tuple[np.ndarray, np.ndarray, dict]:
        R_low = np.zeros(3)
        R_high = np.zeros(3)
        counts = {"low": [], "high": []}
        for k, ticker in enumerate(TICKERS):
            g = df[f"gross_{ticker}"]
            g0, g1 = g[z == 0].dropna(), g[z == 1].dropna()
            R_low[k], R_high[k] = g0.mean(), g1.mean()
            counts["low"].append(len(g0))
            counts["high"].append(len(g1))
        return R_low, R_high, counts

    def _sanity_checks(self, merged: pd.DataFrame, df: pd.DataFrame, z: pd.Series,
                       P: np.ndarray, R_low: np.ndarray, R_high: np.ndarray,
                       Rf: float, counts: dict) -> None:
        print("Date range (merged):", merged["date"].min(), "to", merged["date"].max())
        print("z=0:", (z == 0).sum(), "| z=1:", (z == 1).sum())
        print("P row sums:", P.sum(axis=1))
        print("R_low:", R_low, "| R_high:", R_high, "| Rf:", Rf)
        for ticker, n0, n1 in zip(TICKERS, counts["low"], counts["high"]):
            if n0 < MIN_SAMPLES_WARN or n1 < MIN_SAMPLES_WARN:
                print("WARNING: {} low={} high={}".format(ticker, n0, n1))

    def build_params(self, verbose: bool = False) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
        merged = self._load_and_merge()
        df, z = self._compute_gross_returns_and_align_regime(merged)
        P = self._estimate_P(z)
        R_low, R_high, counts = self._estimate_R_low_R_high(df, z)
        Rf = self.rf_gross
        self._merged, self._returns_df, self._z_aligned, self._counts = merged, df, z, counts
        if verbose:
            self._sanity_checks(merged, df, z, P, R_low, R_high, Rf, counts)
        return P, R_low, R_high, Rf

In [7]:
import numpy as np
from itertools import product
from pathlib import Path
from typing import List, Dict, Tuple, Optional


In [8]:
import numpy as np
from itertools import product
from pathlib import Path
from typing import List, Dict, Tuple, Optional


class MarkovDecisionProcess:

    def __init__(self, P, R_low, R_high, Rf, beta, K, n_assets, risk_aversion):
        self.P = np.asarray(P, dtype=float)
        assert self.P.shape == (2, 2)
        self.R = {
            0: np.asarray(R_low, dtype=float),
            1: np.asarray(R_high, dtype=float),
        }
        self.Rf = float(Rf)
        self.beta = float(beta)
        self.a = float(risk_aversion)
        self.n = int(n_assets)
        assert self.R[0].shape == (self.n,) and self.R[1].shape == (self.n,) and self.a > 0
        self.W = [i / K for i in range(K + 1)]
        self._actions = self._build_feasible_actions()

    def states(self) -> List[int]:
        return [0, 1]

    def actions(self, state: int) -> List[np.ndarray]:
        return self._actions

    def transition_prob(self, state: int, action: np.ndarray, next_state: int) -> float:
        return float(self.P[state, next_state])

    def reward(self, state: int, action: np.ndarray, next_state: int) -> float:
        w = np.asarray(action, dtype=float)
        G = float(np.dot(w, self.R[next_state])) + (1.0 - np.sum(w)) * self.Rf
        return float((1.0 - np.exp(-self.a * G)) / self.a)

    def _build_feasible_actions(self) -> List[np.ndarray]:
        acts: List[np.ndarray] = []
        for w_tuple in product(self.W, repeat=self.n):
            w = np.array(w_tuple, dtype=float)
            if np.sum(w) <= 1.0 + 1e-12:
                acts.append(w)
        return acts


def finite_horizon_dp(mdp: MarkovDecisionProcess, T: int) -> Tuple[List[Dict[int, float]], List[Dict[int, np.ndarray]]]:
    states = mdp.states()
    V: List[Dict[int, float]] = [dict() for _ in range(T + 1)]
    pi: List[Dict[int, np.ndarray]] = [dict() for _ in range(T)]
    for z in states:
        V[T][z] = 0.0
    for t in range(T - 1, -1, -1):
        for z in states:
            best_val, best_action = -np.inf, None
            for a in mdp.actions(z):
                exp_val = sum(
                    mdp.transition_prob(z, a, z_next) * (mdp.reward(z, a, z_next) + mdp.beta * V[t + 1][z_next])
                    for z_next in states
                )
                if exp_val > best_val:
                    best_val, best_action = exp_val, a
            V[t][z] = float(best_val)
            pi[t][z] = np.array(best_action, dtype=float)
    return V, pi


In [9]:
data_dir = Path.cwd() / "data"
estimator = RegimeParamsEstimator(data_dir=data_dir, rf_gross=1.0)
P, R_low, R_high, Rf = estimator.build_params(verbose=False)

mdp = MarkovDecisionProcess(P=P, R_low=R_low, R_high=R_high, Rf=Rf, beta=0.99, K=10, n_assets=len(TICKERS), risk_aversion=1.0)
T = 10
V, pi = finite_horizon_dp(mdp, T=T)

print("V (value by t and regime):")
for t in range(T + 1):
    print("  t={}: z=0 -> {}, z=1 -> {}".format(t, V[t][0], V[t][1]))
print("pi (optimal weights [AAPL,TSLA,MSFT] by t and regime):")
for t in range(T):
    print("  t={}: z=0 -> {}, z=1 -> {}".format(t, pi[t][0], pi[t][1]))

V (value by t and regime):
  t=0: z=0 -> 6.051098351788568, z=1 -> 6.050515770288631
  t=1: z=0 -> 5.472989215063504, z=1 -> 5.472459229711804
  t=2: z=0 -> 4.889040076971078, z=1 -> 4.888563872007405
  t=3: z=0 -> 4.299191935999794, z=1 -> 4.298770722327942
  t=4: z=0 -> 3.703385194402135, z=1 -> 3.703020210190262
  t=5: z=0 -> 3.101559652166108, z=1 -> 3.10125216346061
  t=6: z=0 -> 2.4936545009257416, z=1 -> 2.4934058022789523
  t=7: z=0 -> 1.8796083178099523, z=1 -> 1.8794197329219748
  t=8: z=0 -> 1.2593590592291373, z=1 -> 1.2592319416041262
  t=9: z=0 -> 0.6328440545988606, z=1 -> 0.6327797882160819
  t=10: z=0 -> 0.0, z=1 -> 0.0
pi (optimal weights [AAPL,TSLA,MSFT] by t and regime):
  t=0: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=1: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=2: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=3: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=4: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=5: z=0 -> [0. 1. 0.], z=1 -> [0. 1. 0.]
  t=6: z=0 -> [0. 1. 0.], z=1 -> [0.